In [14]:
import torch
import torch.nn as nn
import torchvision
from av import device
from torch.utils import data
from torchvision import transforms
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader,Subset

from models.cnn import wide_resnet_28, dual_branch_resnet_28
from utils.trainer import Trainer
from models.cnn.dual_branch_resnet_28 import DualBranchResNet28
from models.cnn.resnext_29 import ResNext29
from models.cnn.wide_resnet_28 import WideResNet28

import sys
import os

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
sys.path.append('/content/vision-architectures-pytorch')

## download Data & transform

In [ ]:
!sudo apt-get install -y aria2

!aria2c -x 16 -s 16 https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz -d ./data/

!tar -xvf ./data/cifar-10-python.tar.gz -C ./data/

In [8]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomCrop(32 ,padding= 4),
    transforms.ColorJitter(brightness= 0.2,contrast= 0.2),
    transforms.RandomRotation(degrees= 15),
    transforms.ToTensor(),
    transforms.Normalize((.5,.5,.5),(.5,.5,.5))

])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((.5,.5,.5),(.5,.5,.5))
])

trainset = torchvision.datasets.CIFAR10(
    root = "./data",
    train = True,
    download= False,
    transform= train_transform

)

testset = torchvision.datasets.CIFAR10(
    root = "./data",
    train = False,
    download= False,
    transform= eval_transform

)

valset = trainset = torchvision.datasets.CIFAR10(
    root = "./data",
    train = True,
    download= False,
    transform= train_transform

)


RuntimeError: Dataset not found or corrupted. You can use download=True to download it

In [ ]:
indices = list(range(len(trainset)))
train_ind , val_ind = train_test_split(indices, 0.1,random_state=42 ,stratify=True)

In [ ]:
train_subset = Subset(trainset,train_ind)
val_subset = Subset(valset,val_ind)

In [ ]:
train_loader = DataLoader(
    train_subset,
    batch_size= 128,
    shuffle= True,
    num_workers=2,
    pin_memory= True

)

val_loader = DataLoader(
    val_subset,
    batch_size= 128,
    shuffle= False,
    num_workers=2,
    pin_memory= True

)

train_loader = DataLoader(
    trainset,
    batch_size= 128,
    shuffle= False,
    num_workers=2,
    pin_memory= True

)

In [9]:
wideresnet28 = WideResNet28(3,2,28,10).to(device)
resnext29 = ResNext29(3,10,29,2).to(device)
dualbranchresnet28 = DualBranchResNet28(3,28,10).to(device)

In [10]:
optimizer_wr28 = torch.optim.AdamW(wideresnet28.parameters(),lr=0.0001,weight_decay=5e-4)
optimizer_rn29 = torch.optim.AdamW(resnext29.parameters(),lr=0.0001,weight_decay=5e-4)
optimizer_dr28 = torch.optim.AdamW(dualbranchresnet28.parameters(),lr=0.0001,weight_decay=5e-4)


In [13]:
criterion = nn.CrossEntropyLoss()

In [15]:
train_wr28 =  Trainer(wideresnet28,optimizer_wr28,criterion,device)
train_rn29 =  Trainer(resnext29,optimizer_rn29,criterion,device)
train_dr28 =  Trainer(dualbranchresnet28,optimizer_dr28,criterion,device)

In [ ]:
history_wr28=train_wr28.fit(train_loader,val_loader,70,"wr28")